In [1]:
import numpy as np
import pandas as pd
import re
from sklearn.neighbors import BallTree

## Inspect and clean FAULTS dataset

In [2]:
faults_df = pd.read_csv(filepath_or_buffer="../data/J1939Faults.csv", low_memory=False)
faults_df

,RecordID,ESS_Id,EventTimeStamp,eventDescription,actionDescription,ecuSoftwareVersion,ecuSerialNumber,ecuModel,ecuMake,ecuSource,spn,fmi,active,activeTransitionCount,faultValue,EquipmentID,MCTNumber,Latitude,Longitude,LocationTimeStamp
0,1,990349,2015-02-21 10:47:13.000,Low (Severity Low) Engine Coolant Level,NaN,unknown,unknown,unknown,unknown,0,111,17,True,2,NaN,1439,105354361,38.857638,-84.626851,2015-02-21 11:34:25.000
1,2,990360,2015-02-21 11:34:34.000,NaN,NaN,unknown,unknown,unknown,unknown,11,629,12,True,127,NaN,1439,105354361,38.857638,-84.626851,2015-02-21 11:35:10.000
2,3,990364,2015-02-21 11:35:31.000,Incorrect Data Steering Wheel Angle,NaN,unknown,unknown,unknown,unknown,11,1807,2,False,127,NaN,1369,105336226,41.421250,-87.767361,2015-02-21 11:35:26.000
3,4,990370,2015-02-21 11:35:33.000,Incorrect Data Steering Wheel Angle,NaN,unknown,unknown,unknown,unknown,11,1807,2,True,127,NaN,1369,105336226,41.421018,-87.767361,2015-02-21 11:36:08.000
4,5,990416,2015-02-21 11:39:41.000,NaN,NaN,22281684P01*22357957P01*22362082P01*,13063430,0USA13_13_0415_2238A,VOLVO,0,4364,17,False,2,NaN,1674,105427130,38.416481,-89.442638,2015-02-21 11:39:37.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1187330,1248454,123904424,2020-03-06 14:00:26.000,Low (Severity Low) Catalyst Tank Level,NaN,04384413*22383729*082218154102*60701732*G1*BGT*,80156139,6X1u17D1500000000,CMMNS,0,1761,17,False,3,NaN,2282,105439740,37.094768,-85.897407,2020-03-06 14:00:21.000
1187331,1248455,123905139,2020-03-06 14:04:23.000,Condition Exists Engine Protection Torque Derate,NaN,04358814*06099720*030816202706*09400153*G1*BDR*,79932020,6X1u13D1500000000,CMMNS,0,1569,31,True,5,NaN,1994,105354084,34.390740,-79.461805,2020-03-06 14:04:59.000
1187332,1248456,123905996,2020-03-06 14:13:38.000,Abnormal Rate of Change Aftertreatment 1 Intak...,NaN,05317106*05100987*050719120655*09401585*G1*BDR*,79880653,6X1u13D1500000000,CMMNS,0,3216,10,True,1,NaN,1850,105336308,34.430370,-84.920509,2020-03-06 14:14:14.000
1187333,1248457,123906113,2020-03-06 14:14:13.000,Low (Severity Medium) Engine Coolant Level,NaN,04384413*22544852*090619141107*60701756*G1*BGT*,NaN,NaN,NaN,0,111,18,True,8,NaN,2377,108605700,35.030925,-85.321527,2020-03-06 14:14:49.000


In [3]:
faults_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1187335 entries, 0 to 1187334
Data columns (total 20 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   RecordID               1187335 non-null  int64  
 1   ESS_Id                 1187335 non-null  int64  
 2   EventTimeStamp         1187335 non-null  object 
 3   eventDescription       1126490 non-null  object 
 4   actionDescription      0 non-null        float64
 5   ecuSoftwareVersion     891285 non-null   object 
 6   ecuSerialNumber        844318 non-null   object 
 7   ecuModel               1122577 non-null  object 
 8   ecuMake                1122577 non-null  object 
 9   ecuSource              1187335 non-null  int64  
 10  spn                    1187335 non-null  int64  
 11  fmi                    1187335 non-null  int64  
 12  active                 1187335 non-null  bool   
 13  activeTransitionCount  1187335 non-null  int64  
 14  faultValue        

### Possible Features:
* RecordID is unique
* EventTimeStamp has 1,050,909 unique values (no nan)
* eventDescription has 60,845 nan (Create 'severity' column based on descriptions)
* ecuSoftwareVersion has 1,899 unique values (296,050 nan)
* ecuModel has 30 unique values (64,758 nan)
* ecuMake has 23 unique values (64,758 nan)
* ecuSource has 5 unique values (no nan)
* spn has 450 unique values (no nan)
* fmi has 26 unique values (no nan)
* active has 2 unique values (no nan)
* EquipmentID has 1,927 unique values (no nan)
* MCTNumber has 768 unique values (no nan)
* Latitude has 211,823 unique values (no nan)
* Longitude has 265,211 unique values (no nan)
* LocationTimeStamp has 1,036,006 unique values (no nan)

### Drop Columns:
* ESS_Id
* actionDescription
* ecuSerialNumber
* faultValue

In [4]:
# Drop unnecessary columns
faults_df = faults_df.drop(columns=['ESS_Id', 'actionDescription', 'ecuSerialNumber', 'faultValue'])

In [5]:
# Convert timestamps to datetime objects
faults_df['EventTimeStamp'] = pd.to_datetime(faults_df['EventTimeStamp'])
faults_df['LocationTimeStamp'] = pd.to_datetime(faults_df['LocationTimeStamp'])

In [6]:
# Create column for active codes that are near service stations
service_stations = [
    (36.0666667, -86.4347222),
    (35.5883333, -86.4438888),
    (36.1950, -83.174722)
]

earth_radius_km = 6371.0
    
station_radians = np.radians(service_stations)
points_radians = np.radians(faults_df[['Latitude', 'Longitude']].values)
    
tree = BallTree(station_radians, metric='haversine')
    
# Query radius in radians
indices = tree.query_radius(X=points_radians, r=1.0 / earth_radius_km)
    
faults_df['NearServiceStation'] = np.array([len(idx) > 0 for idx in indices])
faults_df['NearServiceStation'].value_counts()

NearServiceStation
False    1055968
True      131367
Name: count, dtype: int64

In [7]:
# Create column for full derate flag
# Full derate is determined by spn code 5246 and engine code active is true
faults_df['IsFullDerate'] = (
        (faults_df['spn'] == 5246)
        & faults_df['active']
        & ~faults_df['NearServiceStation']
    )
faults_df['IsFullDerate'].value_counts()

IsFullDerate
False    1186837
True         498
Name: count, dtype: int64

In [8]:
# Create columns for severity level from eventDescription
def extract_severity(text):

    if pd.isna(text):
        return np.nan

    # Severity with "Low", "Medium", or "high"
    pattern = r'Severity\s+(Low|Medium|High)'

    # Find pattern
    match = re.search(pattern, text)

    if match: 
        return f"Severity {match.group(1)}"
    else: 
        return np.nan

faults_df['Severity_Level'] = faults_df['eventDescription'].apply(extract_severity)

severity_map = {
    'Severity Low': 1,
    'Severity Medium': 2,
    'Severity High': 3
}

faults_df['Severity_Level_Numeric'] = faults_df['Severity_Level'].map(severity_map)

In [9]:
# Create column for full derate in 8 hour window flag
faults_df = faults_df.sort_values(['EquipmentID', 'EventTimeStamp'])

# Initialize target column
faults_df['DeratePredictionTarget'] = 0

# Dataframe with just the derate events
derate_events = faults_df[faults_df['IsFullDerate']].copy()

# Group by EquipmentID 
for equipment_id, group in faults_df.groupby('EquipmentID'):
    # Get derate events for this truck only
    truck_derates = derate_events[derate_events['EquipmentID'] == equipment_id]
    
    if len(truck_derates) > 0:
        # Get indices and timestamps for this truck's rows
        truck_indices = group.index
        truck_timestamps = group['EventTimeStamp'].values
        
        # For each derate event in this truck
        for _, derate_row in truck_derates.iterrows():
            derate_time = derate_row['EventTimeStamp']
            
            # Window: 
            window_start = derate_time - pd.Timedelta(hours=8)
            window_end = derate_time - pd.Timedelta(hours=.001)
            
            # All events in the prediction window
            in_window = (truck_timestamps >= window_start) & (truck_timestamps <= window_end)
            indices_to_mark = truck_indices[in_window]
            
            # Marked as predicting a derate
            faults_df.loc[indices_to_mark, 'DeratePredictionTarget'] = 1

print(f"Total events: {len(faults_df)}")
print(f"Events predicting a derate: {faults_df['DeratePredictionTarget'].sum()}")

Total events: 1187335
Events predicting a derate: 1690


### Possibly need to remove inactive records from DeratePredictionTarget

## Inspect DIAGNOSTICS

In [10]:
diagnostics_df = pd.read_csv(filepath_or_buffer="../data/VehicleDiagnosticOnboardData.csv", low_memory=False)
diagnostics_df

,Id,Name,Value,FaultId
0,1,IgnStatus,False,1
1,2,EngineOilPressure,0,1
2,3,EngineOilTemperature,96.74375,1
3,4,TurboBoostPressure,0,1
4,5,EngineLoad,11,1
...,...,...,...,...
12821621,12864020,EngineCoolantTemperature,181.4,1248457
12821622,12864021,ParkingBrake,False,1248457
12821623,12864022,SwitchedBatteryVoltage,14.1,1248457
12821624,12864023,DistanceLtd,28606.65625,1248457


In [11]:
diagnostics_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12821626 entries, 0 to 12821625
Data columns (total 4 columns):
 #   Column   Dtype 
---  ------   ----- 
 0   Id       int64 
 1   Name     object
 2   Value    object
 3   FaultId  int64 
dtypes: int64(2), object(2)
memory usage: 391.3+ MB


In [12]:
# To get the on-board diagnostics at the time of the fault code, we can match the **RecordID** to the **FaultId**.
diagnostics_df.loc[diagnostics_df['FaultId'] == 1]

,Id,Name,Value,FaultId
0,1,IgnStatus,False,1
1,2,EngineOilPressure,0,1
2,3,EngineOilTemperature,96.74375,1
3,4,TurboBoostPressure,0,1
4,5,EngineLoad,11,1
5,6,AcceleratorPedal,0,1
6,7,IntakeManifoldTemperature,78.8,1
7,8,FuelRate,0,1
8,9,FuelLtd,12300.907429328,1
9,10,EngineRpm,0,1


In [13]:
diagnostics_df.loc[diagnostics_df['FaultId'] == 46]

,Id,Name,Value,FaultId
418,419,IgnStatus,True,46
419,420,LampStatus,22527,46


## Inpsect SERVICE FAULT CODES

In [14]:
sfc_df = pd.read_excel(io="../data/Service Fault Codes_1_0_0_167.xlsx")
sfc_df

C:\ProgramData\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


,Published in CES 14602,Cummins Fault Code,Revision,PID,SID,MID,J1587 FMI,SPN,J1939 FMI,J2012 Pcode,Lamp Color,Lamp Device,Cummins Description,Algorithm Description
0,Y,111,167,Not Mapped,254,0,12,629,12,P0606,Red,Stop / Shutdown,Engine Control Module Critical Internal Failur...,Error internal to the ECM related to memory ha...
1,Y,112,167,Not Mapped,20,128,7,635,7,Not Mapped,Red,Stop / Shutdown,Engine Timing Actuator Driver Circuit - Mechan...,Mechanical failure in the engine timing actuat...
2,Y,113,167,Not Mapped,20,128,3,635,3,Not Mapped,Amber,Warning,Engine Timing Actuator Driver Circuit - Voltag...,High signal voltage detected at the engine tim...
3,Y,114,167,Not Mapped,20,128,4,635,4,Not Mapped,Amber,Warning,Engine Timing Actuator Driver Circuit - Voltag...,Low voltage detected at the engine timing actu...
4,Y,115,167,190,Not Mapped,Not Mapped,2,612,2,P0008,Red,Stop / Shutdown,Engine Magnetic Speed/Position Lost Both of Tw...,The ECM has detected that the primary and back...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7119,Y,9996,167,Not Mapped,155,0,11,524286,31,Not Mapped,Amber,Warning,Reserved for temporary use - Condition Exists,NaN
7120,Y,9997,167,Not Mapped,155,0,11,524286,31,Not Mapped,Amber,Warning,Reserved for temporary use - Condition Exists,NaN
7121,Y,9998,167,Not Mapped,155,0,11,524286,31,Not Mapped,Amber,Warning,Reserved for temporary use - Condition Exists,NaN
7122,Y,9999,167,Not Mapped,155,0,11,524286,31,Not Mapped,Amber,Warning,Reserved for temporary use - Condition Exists,NaN


In [15]:
sfc_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7124 entries, 0 to 7123
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Published in CES 14602  7124 non-null   object
 1   Cummins Fault Code      7124 non-null   int64 
 2   Revision                7124 non-null   int64 
 3   PID                     7124 non-null   object
 4   SID                     7124 non-null   object
 5   MID                     7124 non-null   object
 6   J1587 FMI               7124 non-null   int64 
 7   SPN                     7124 non-null   int64 
 8   J1939 FMI               7124 non-null   int64 
 9   J2012 Pcode             7124 non-null   object
 10  Lamp Color              6253 non-null   object
 11  Lamp Device             6253 non-null   object
 12  Cummins Description     7124 non-null   object
 13  Algorithm Description   2005 non-null   object
dtypes: int64(5), object(9)
memory usage: 779.3+ KB


In [16]:
# Inspect codes for full derates
(
    sfc_df
    .loc[sfc_df['SPN'] == 5246]
    .loc[sfc_df['J1939 FMI'].isin([ 0, 15, 16, 19, 14])]
)

,Published in CES 14602,Cummins Fault Code,Revision,PID,SID,MID,J1587 FMI,SPN,J1939 FMI,J2012 Pcode,Lamp Color,Lamp Device,Cummins Description,Algorithm Description
2518,Y,3712,167,Not Mapped,Not Mapped,Not Mapped,0,5246,0,Not Mapped,Red,Stop / Shutdown,Aftertreatment SCR Operator Inducement - Data ...,SCR inducement of 5 mph derate - Fault Code 41...
2781,Y,4134,167,Not Mapped,Not Mapped,Not Mapped,0,5246,15,Not Mapped,Amber,Warning,Aftertreatment SCR Operator Inducement - Data ...,SCR inducement - Least Severe - Fault Code 371...
4338,Y,6254,167,Not Mapped,Not Mapped,Not Mapped,0,5246,16,Not Mapped,Amber,Warning,Aftertreatment SCR Operator Inducement Severit...,NaN


In [17]:
(
    sfc_df
    .loc[sfc_df['SPN'] == 629]
    .loc[sfc_df['J1939 FMI'] == 12]
)

,Published in CES 14602,Cummins Fault Code,Revision,PID,SID,MID,J1587 FMI,SPN,J1939 FMI,J2012 Pcode,Lamp Color,Lamp Device,Cummins Description,Algorithm Description
0,Y,111,167,Not Mapped,254,0,12,629,12,P0606,Red,Stop / Shutdown,Engine Control Module Critical Internal Failur...,Error internal to the ECM related to memory ha...
180,Y,343,167,Not Mapped,254,0,12,629,12,P0607,Amber,Warning,Engine Control Module Warning Internal Hardwar...,ECM power supply errors have been detected.
689,Y,1116,167,Not Mapped,254,0,12,629,12,Not Mapped,Amber,Warning,Engine Control Module Critical Internal Failur...,ECM Internal failure has occurred.
854,Y,1388,167,Not Mapped,254,0,12,629,12,Not Mapped,NaN,NaN,Engine Control Module Data Lost - Bad Intellig...,The ECM data has been lost.
1019,Y,1597,167,Not Mapped,254,0,12,629,12,Not Mapped,Maintenance,Maintenance,Engine Control Module Critical Internal Failur...,The ECM has occurred an internal failure.


## Merge FAULTS and DIAGNOSTICS

### Diagnostics 1-to-1 with faults

In [18]:
# Pivot DIAGNOSTICS wider
diagnostics_pivot_wider_df = diagnostics_df.pivot(
    columns='Name',
    index='FaultId',
    values='Value'
)
diagnostics_pivot_wider_df

Name,AcceleratorPedal,BarometricPressure,CruiseControlActive,CruiseControlSetSpeed,DistanceLtd,EngineCoolantTemperature,EngineLoad,EngineOilPressure,EngineOilTemperature,EngineRpm,...,FuelTemperature,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure
FaultId,,,,,,,,,,,,,,,,,,,,,
1,0,14.21,False,66.48672,423178.7,100.4,11,0,96.74375,0,...,NaN,False,78.8,1023,True,NaN,0,3276.75,NaN,0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,16639,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1248454,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1023,NaN,NaN,NaN,NaN,NaN,NaN
1248455,100,14.5,True,64.6226,423937.9,185,51,37.12,211.4937,1310.25,...,32,True,98.6,18431,False,NaN,65.01096,NaN,73.2,7.83
1248456,0,14.355,True,66.48672,465925.4,186.8,62,41.18,212.8438,1340.75,...,NaN,True,91.4,17407,NaN,NaN,66.5741,NaN,100,6.96


In [19]:
# Replace commas with decimal points and convert to floats
float_columns = ['AcceleratorPedal', 'BarometricPressure', 'DistanceLtd', 'EngineCoolantTemperature', 'EngineOilPressure', 'EngineOilTemperature', 'EngineRpm', 'EngineTimeLtd', 'FuelLevel', 'FuelLtd', 'FuelRate', 'FuelTemperature', 'IntakeManifoldTemperature', 'Speed', 'SwitchedBatteryVoltage', 'Throttle', 'TurboBoostPressure']

for col in float_columns:
    print(col)
    diagnostics_pivot_wider_df[col] = diagnostics_pivot_wider_df[col].str.replace(pat=',', repl='.').astype(float)

AcceleratorPedal
BarometricPressure
DistanceLtd
EngineCoolantTemperature
EngineOilPressure
EngineOilTemperature
EngineRpm
EngineTimeLtd
FuelLevel
FuelLtd
FuelRate
FuelTemperature
IntakeManifoldTemperature
Speed
SwitchedBatteryVoltage
Throttle
TurboBoostPressure


In [20]:
diagnostics_pivot_wider_df.to_csv('../data/diagnostics_pivot_wider.csv', index=False)

In [21]:
faults_diagnostics_df = pd.merge(
    left=faults_df,
    right=diagnostics_pivot_wider_df,
    how='inner',
    left_on='RecordID',
    right_on='FaultId',
    validate='1:1'
)
faults_diagnostics_df

,RecordID,EventTimeStamp,eventDescription,ecuSoftwareVersion,ecuModel,ecuMake,ecuSource,spn,fmi,active,...,FuelTemperature,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure
0,1046062,2018-08-15 09:55:05,Incorrect Data J1939 Network #1 Primary Vehicl...,unknown,unknown,unknown,11,639,2,True,...,32.0,True,86.0,1279,False,NaN,4.786500,NaN,100.0,0.87
1,1019516,2011-01-01 01:09:24,Abnormal Update Rate Transmission Output Shaft...,04384413*22161683*121817205924*60701721*G1*BGT*,6X1u17D1500000000,CMMNS,0,191,9,True,...,NaN,True,89.6,17407,True,NaN,0.000000,NaN,100.0,0.00
2,1019476,2011-01-01 02:03:05,Abnormal Update Rate Transmission Output Shaft...,04384413*22161683*121817205924*60701721*G1*BGT*,6X1u17D1500000000,CMMNS,0,191,9,True,...,NaN,True,91.4,17407,True,NaN,0.000000,NaN,100.0,0.00
3,1019409,2011-01-01 03:36:42,Low (Severity Medium) Engine Coolant Level,04384413*22161683*121817205924*60701721*G1*BGT*,6X1u17D1500000000,CMMNS,0,111,18,True,...,NaN,True,87.8,2047,True,NaN,0.000000,NaN,100.0,0.00
4,1019410,2011-01-01 03:41:34,Low (Severity Medium) Engine Coolant Level,04384413*22161683*121817205924*60701721*G1*BGT*,6X1u17D1500000000,CMMNS,0,111,18,False,...,NaN,NaN,NaN,1023,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1187330,4495,2015-02-24 16:24:05,Low (Severity Medium) Catalyst Tank Level,05317106*04119044*051914190353*09400015*G1*BDR*,6X1u13D1500000000,CMMNS,0,1761,18,False,...,NaN,NaN,NaN,1023,NaN,NaN,NaN,NaN,NaN,NaN
1187331,6439,2015-02-26 13:12:11,NaN,05317106*04119044*051914190353*09400015*G1*BDR*,6X1u13D1500000000,CMMNS,0,5848,9,True,...,32.0,True,84.2,17407,False,NaN,2.058292,3276.75,0.0,0.29
1187332,6447,2015-02-26 13:50:59,NaN,05317106*04119044*051914190353*09400015*G1*BDR*,6X1u13D1500000000,CMMNS,0,5848,9,False,...,NaN,NaN,NaN,1023,NaN,NaN,NaN,NaN,NaN,NaN
1187333,4953,2015-02-25 06:08:43,Incorrect Data J1939 Network #1 Primary Vehicl...,unknown,unknown,unknown,11,639,2,True,...,32.0,True,93.2,1279,False,NaN,4.378725,3276.75,0.0,1.74


In [22]:
faults_diagnostics_df.to_csv('../data/faults_diagnostics.csv', index=False)

In [23]:
# Split data into training and testing
cutoff_date = '2018-12-31 23:59:59'

training_faults_diagnostics_df = faults_diagnostics_df[faults_diagnostics_df['EventTimeStamp'] <= cutoff_date]
testing_faults_diagnostics_df = faults_diagnostics_df[faults_diagnostics_df['EventTimeStamp'] > cutoff_date]

In [24]:
training_faults_diagnostics_df

,RecordID,EventTimeStamp,eventDescription,ecuSoftwareVersion,ecuModel,ecuMake,ecuSource,spn,fmi,active,...,FuelTemperature,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure
0,1046062,2018-08-15 09:55:05,Incorrect Data J1939 Network #1 Primary Vehicl...,unknown,unknown,unknown,11,639,2,True,...,32.0,True,86.0,1279,False,NaN,4.786500,NaN,100.0,0.87
1,1019516,2011-01-01 01:09:24,Abnormal Update Rate Transmission Output Shaft...,04384413*22161683*121817205924*60701721*G1*BGT*,6X1u17D1500000000,CMMNS,0,191,9,True,...,NaN,True,89.6,17407,True,NaN,0.000000,NaN,100.0,0.00
2,1019476,2011-01-01 02:03:05,Abnormal Update Rate Transmission Output Shaft...,04384413*22161683*121817205924*60701721*G1*BGT*,6X1u17D1500000000,CMMNS,0,191,9,True,...,NaN,True,91.4,17407,True,NaN,0.000000,NaN,100.0,0.00
3,1019409,2011-01-01 03:36:42,Low (Severity Medium) Engine Coolant Level,04384413*22161683*121817205924*60701721*G1*BGT*,6X1u17D1500000000,CMMNS,0,111,18,True,...,NaN,True,87.8,2047,True,NaN,0.000000,NaN,100.0,0.00
4,1019410,2011-01-01 03:41:34,Low (Severity Medium) Engine Coolant Level,04384413*22161683*121817205924*60701721*G1*BGT*,6X1u17D1500000000,CMMNS,0,111,18,False,...,NaN,NaN,NaN,1023,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1187330,4495,2015-02-24 16:24:05,Low (Severity Medium) Catalyst Tank Level,05317106*04119044*051914190353*09400015*G1*BDR*,6X1u13D1500000000,CMMNS,0,1761,18,False,...,NaN,NaN,NaN,1023,NaN,NaN,NaN,NaN,NaN,NaN
1187331,6439,2015-02-26 13:12:11,NaN,05317106*04119044*051914190353*09400015*G1*BDR*,6X1u13D1500000000,CMMNS,0,5848,9,True,...,32.0,True,84.2,17407,False,NaN,2.058292,3276.75,0.0,0.29
1187332,6447,2015-02-26 13:50:59,NaN,05317106*04119044*051914190353*09400015*G1*BDR*,6X1u13D1500000000,CMMNS,0,5848,9,False,...,NaN,NaN,NaN,1023,NaN,NaN,NaN,NaN,NaN,NaN
1187333,4953,2015-02-25 06:08:43,Incorrect Data J1939 Network #1 Primary Vehicl...,unknown,unknown,unknown,11,639,2,True,...,32.0,True,93.2,1279,False,NaN,4.378725,3276.75,0.0,1.74


In [25]:
training_faults_diagnostics_df.to_csv('../data/training_faults_diagnostics.csv', index=False)
testing_faults_diagnostics_df.to_csv('../data/testing_faults_diagnostics.csv', index=False)